# Extraction of medications on admission from clinical notes

## Extraction run over the experiment grid

- Each note of the selected environment is processed once per `(model, strategy)` cell:
  4 models x 3 prompting strategies.
- All input variables (paths, prompt files, grid, Ollama runtime settings) are defined
  **here** and passed as parameters. `utils/llm/llm_extraction.py` holds pure logic only.
- The model sits on the **outer** loop, so it is loaded into VRAM once per cell instead
  of being swapped in and out.
- Notes inside a cell run concurrently: `ExtractionRunner` is frozen and stateless, and
  each call writes its own file, so threads never contend.
- Caching in `ExtractionRunner.extract()` makes this safely re-runnable after an
  interruption without repeating completed calls.

## Imports

In [ ]:
import json
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm

from clinical_notes_extraction.config import PROJECT_ROOT
from clinical_notes_extraction.utils.llm.llm_extraction import ExtractionRunner

## Configuration

`ENVIRONMENT` selects the phase data:

- `dev` for model/strategy selection -- the winning combination is chosen on this data;
- `prod` only for the final one-shot run of that combination on the held-out test set.

`STRATEGIES` drives the whole notebook: which prompt shells are read, which example
assets are loaded and which grid cells are executed.

`MAX_WORKERS` must not exceed the server's `OLLAMA_NUM_PARALLEL`; above that, requests
just queue. Set it to `1` whenever `DEBUG_PROMPT` is on, since concurrent prints
interleave into unreadable output.

In [ ]:
# Phase: "dev" for selection, "prod" for the final held-out run.
ENVIRONMENT = "dev"

# Strategies under test. Uncomment to run the full grid.
STRATEGIES = ["zero_shot"]  # , "few_shot", "dynamic"

# Concurrent notes per (model, strategy) cell.
MAX_WORKERS = 4

# Dump the fully assembled prompt of every call. Requires MAX_WORKERS = 1.
DEBUG_PROMPT = False

ROOT = PROJECT_ROOT / "scripts" / "3_information_extraction" / "3_2_medications_on_admission"
CONFIG_DIR = PROJECT_ROOT / "config"
PROMPTS_DIR = ROOT / "prompts"
DATA_DIR = ROOT / "data"
RESULTS_DIR = DATA_DIR / "llm_extraction_results" / ENVIRONMENT

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if DEBUG_PROMPT and MAX_WORKERS > 1:
    raise ValueError("DEBUG_PROMPT requires MAX_WORKERS = 1")

## Load configuration and split

`runnable_models.json` carries the runnable model tags and the Ollama block
(`url`, `options`, `timeout_seconds`).
The split holds `note_id` and `text`; `cluster` is only read when the dynamic strategy
is active, since it is the key that selects the medoid example.

In [ ]:
config = json.loads((CONFIG_DIR / "runnable_models.json").read_text(encoding="utf-8"))
models = config["models"]
ollama_config = config["ollama"]

# "cluster" is only needed by the dynamic strategy to look up the medoid example.
columns = ["note_id", "text"] + (["cluster"] if "dynamic" in STRATEGIES else [])

env_df = pd.read_parquet(DATA_DIR / f"{ENVIRONMENT}_sample.parquet")
notes = env_df[columns].to_dict("records")

# TEMPORARY: smoke test before the full grid. Remove both lines afterwards.
# models = ["deepseek-r1:70b"]
# notes = notes[:5]

n_cells = len(models) * len(STRATEGIES) * len(notes)
print(f"{len(models)} models x {len(STRATEGIES)} strategies x {len(notes)} notes = {n_cells} cells")

## Load prompt assets once

Reading these here avoids re-reading the same files on every grid cell.

- `role.md` is the system prompt;
- the strategy templates are the user prompt shells, read only for the active strategies;
- `expected_template.md` is the shared output schema injected into every shell.

In [ ]:
# System prompt (persona) and the shared expected-output schema.
role = (PROMPTS_DIR / "role.md").read_text(encoding="utf-8")
expected_template = (PROMPTS_DIR / "expected_template.md").read_text(encoding="utf-8")

# One user-prompt shell per active strategy, keyed by strategy name.
strategy_templates = {
    strategy: (PROMPTS_DIR / f"{strategy}.md").read_text(encoding="utf-8")
    for strategy in STRATEGIES
}

## Example assets

Loaded only by the strategies that consume them, so a zero-shot run touches no example
file at all.

- **few-shot**: a fixed pool of curated examples, external to the 32-note sample;
- **dynamic**: one annotated medoid per cluster, keyed by cluster id.

Both loaders are expected to yield dicts shaped as `{"text": ..., "medications": [...]}`,
which is what `format_examples` renders.

In [ ]:
# Few-shot: fixed pool of curated examples, external to the 32-note sample.
# few_shot_pool = json.loads(
#     (DATA_DIR / "annotations/few_shot_examples.json").read_text(encoding="utf-8")
# )["examples"]

# Dynamic: one annotated medoid per cluster, keyed by cluster id.
# medoids_df = pd.read_parquet(
#     DATA_DIR / "annotations/dynamic_prompts/dynamic_prompt_medoids_annotations.parquet"
# )
# medoids = {
#     str(row["cluster"]): {
#         "text": row["text"],
#         "medications": json.loads(row["annotation_json"])["medications"],
#     }
#     for row in medoids_df.to_dict("records")
# }

## Example selection per strategy

The only thing that varies across strategies is what gets injected into `{EXAMPLES}`:

- `zero_shot` passes an empty string (the placeholder simply vanishes on `.replace()`);
- `few_shot` passes the fixed pool;
- `dynamic` passes the single medoid of the note's cluster.

`format_examples` renders the example dicts into a text block. It uses `json.dumps` for
the expected output -- literal braces are exactly why prompt injection uses `.replace()`
and never `.format()`.

> Check the delimiter format below against the actual `few_shot.md` / `dynamic.md` so the
> in-prompt examples match the shell around them.

In [ ]:
def format_examples(items: list[dict]) -> str:
    """Render example dicts (text + gold medications) into a prompt block."""
    blocks = []
    for i, example in enumerate(items, start=1):
        output = json.dumps(
            {"medications": example["medications"]}, indent=2, ensure_ascii=False
        )
        blocks.append(
            f"<example {i}>\n"
            f"<note>\n{example['text']}\n</note>\n"
            f"<output>\n{output}\n</output>\n"
            f"</example {i}>"
        )
    return "\n\n".join(blocks)


def examples_for(strategy: str, note: dict) -> str:
    """Return the formatted example block for one (strategy, note) cell."""
    if strategy == "zero_shot":
        return ""
    # if strategy == "few_shot":
    #     return format_examples(few_shot_pool)
    # if strategy == "dynamic":
    #     # The note's cluster (known from clustering) selects its medoid.
    #     return format_examples([medoids[str(note["cluster"])]])
    raise ValueError(f"Unknown strategy: {strategy}")

## Run directory and logging

Every execution gets its own timestamped folder under `RESULTS_DIR`, so results, the run
log and the frozen configuration always travel together and no previous run is ever
overwritten:

```
<RESULTS_DIR>/<YYYYMMDD_HHMMSS>/
```

The timestamp sorts chronologically as a plain string, so no scan of existing folders is
needed to pick the next name.

In [ ]:
def setup_logging(run_dir: Path) -> logging.Logger:
    logger = logging.getLogger("extraction")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()  # avoid duplicate handlers when re-running the cell
    fmt = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"
    )
    for handler in (
        logging.FileHandler(run_dir / "run.log", encoding="utf-8"),  # to file
        logging.StreamHandler(),                                     # to console
    ):
        handler.setFormatter(fmt)
        logger.addHandler(handler)
    return logger


run_started = datetime.now()
run_date = run_started.date().isoformat()

run_dir = RESULTS_DIR / run_started.strftime("%Y%m%d_%H%M%S")
run_dir.mkdir(parents=True, exist_ok=True)

logger = setup_logging(run_dir)
logger.info(f"Run dir: {run_dir}")

(run_dir / "config.json").write_text(
    json.dumps(
        {
            "environment": ENVIRONMENT,
            "run_timestamp": run_started.isoformat(timespec="seconds"),
            "run_date": run_date,
            "models": models,
            "strategies": STRATEGIES,
            "n_notes": len(notes),
            "max_workers": MAX_WORKERS,
            "ollama_config": ollama_config,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

## Preflight

`extract()` deliberately lets connection errors and timeouts propagate: if the server is
down, stopping loudly is the correct behaviour. This check fails in seconds instead of
letting the first note discover it, and confirms every model in the grid is actually
pulled before a multi-hour run starts.

In [ ]:
response = requests.get(f"{ollama_config['url']}/api/tags", timeout=5)
response.raise_for_status()
available = {model["name"] for model in response.json()["models"]}

missing = [model for model in models if model not in available]
if missing:
    raise RuntimeError(f"Models not pulled in Ollama: {missing}")

logger.info(f"Ollama reachable at {ollama_config['url']} | models available: {models}")

## Run the grid

Every `(model, strategy, note)` cell goes through `ExtractionRunner.extract`, which
assembles the prompt, calls Ollama in JSON mode, validates with Pydantic and writes the
record to disk. Because each cell caches its own result, rerunning this loop after an
interruption only fills in what is missing.

`runner.load()` pays the cold load once per cell, outside the per-call `timeout_seconds`,
which is sized for generation and not for pulling a 70b into VRAM.

Failure handling belongs to `extract()`, not to this loop:

- model-side failures (`invalid`) and infrastructure failures (`error`) are **returned**
  as a record and persisted per model and per strategy under
  `<run_dir>/<model>/errors/<strategy>/<note_id>.json`, keeping the raw response;
- connection errors, timeouts and unreplaced placeholders **raise**, cancel the queued
  notes and abort the run -- catching them here would silently grind the whole grid
  against a dead server.

In [ ]:
records: list[dict] = []

for model in models:
    runner = None
    try:
        for strategy in STRATEGIES:
            runner = ExtractionRunner(
                model=model,
                strategy=strategy,
                role=role,
                template=strategy_templates[strategy],
                expected_template=expected_template,
                results_dir=run_dir,
                ollama_config=ollama_config,
                run_date=run_date,
                debug_prompt=DEBUG_PROMPT,
            )

            logger.info(
                f"Start: model={model} strategy={strategy} "
                f"notes={len(notes)} workers={MAX_WORKERS}"
            )
            # Cold load once per model: a no-op after the first strategy, since
            # keep_alive holds the weights in VRAM across the whole cell.
            runner.load()

            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
                futures = [
                    executor.submit(
                        runner.extract,
                        note_id=note["note_id"],
                        note_text=note["text"],
                        examples=examples_for(strategy, note),
                    )
                    for note in notes
                ]
                try:
                    for future in tqdm(
                        as_completed(futures),
                        total=len(futures),
                        desc=f"{model} | {strategy}",
                    ):
                        record = future.result()
                        records.append(record)

                        if record["status"] != "ok":
                            logger.warning(
                                f"{record['status']}: model={model} strategy={strategy} "
                                f"note_id={record['note_id']} error={record['error']}"
                            )
                except BaseException:
                    # A raised failure invalidates every remaining cell: drop the
                    # queue instead of waiting for it to drain against a dead server.
                    executor.shutdown(cancel_futures=True)
                    raise

            logger.info(f"End: model={model} strategy={strategy}")
    finally:
        # Only between models: keeping the weights resident across strategies
        # is the point of keep_alive.
        if runner is not None:
            runner.unload()

## Run summary

Two tables, both written next to the results so the run is auditable without re-walking
the `errors/` tree.

**Status** -- counts per `(model, strategy)` cell:

- `ok` -- validated against the Pydantic schema;
- `invalid` -- the model answered but the output is malformed or off-schema (recall = 0);
- `error` -- infrastructure failure (non-2xx, error payload, exhausted OOM retry).

**Cost** -- tokens and latency per cell, plus `context_overflow`: the number of notes
whose prompt and completion together hit `num_ctx`. Any value above zero means Ollama
silently truncated the START of those prompts, dropping the role and the schema, so
their scores are not interpretable and `num_ctx` has to be raised before trusting them.

In [ ]:
runs = pd.DataFrame(records).reindex(
    columns=["model", "strategy", "note_id", "status", "error"]
)

status_summary = (
    runs.pivot_table(
        index=["model", "strategy"],
        columns="status",
        values="note_id",
        aggfunc="count",
        fill_value=0,
    )
    .reindex(columns=["ok", "invalid", "error"], fill_value=0)
    .reset_index()
)
status_summary["total"] = status_summary[["ok", "invalid", "error"]].sum(axis=1)
status_summary.to_csv(run_dir / "status_summary.csv", index=False)

# usage is None on records that never reached the server, hence the empty-dict fallback.
usage = pd.DataFrame(
    [
        {"model": record["model"], "strategy": record["strategy"], **(record["usage"] or {})}
        for record in records
    ]
).reindex(
    columns=[
        "model",
        "strategy",
        "prompt_tokens",
        "completion_tokens",
        "total_duration_s",
        "tokens_per_second",
        "context_overflow",
    ]
)

cost_summary = (
    usage.groupby(["model", "strategy"], dropna=False)
    .agg(
        prompt_tokens=("prompt_tokens", "mean"),
        completion_tokens=("completion_tokens", "mean"),
        seconds_per_note=("total_duration_s", "mean"),
        tokens_per_second=("tokens_per_second", "mean"),
        context_overflow=("context_overflow", "sum"),
    )
    .round(1)
    .reset_index()
)
cost_summary.to_csv(run_dir / "cost_summary.csv", index=False)

failures = runs[runs["status"] != "ok"]
failures.to_json(run_dir / "failures.json", orient="records", indent=2, force_ascii=False)

logger.info(f"Done: {len(records)} cells, {len(failures)} failed -> {run_dir}")

display(status_summary)
display(cost_summary)